In [ ]:
import os
import json
import time
import random
# from kaggle_secrets import UserSecretsClient # Removed due to ModuleNotFoundError in Colab
from google import genai
from google.genai import types

from google.colab import userdata

# 1. Initialize API Client
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
# 2. Automated Scenario Seed Lists (Randomized to ensure high variety)
INJURY_SEEDS = [
"sharp anterior shoulder pinch at bottom of flat bench press",
"patellar tendon ache right below kneecap during heavy barbell squats",
"lumbar spine compression and tightness on conventional deadlift lockout",
"medial elbow throbbing (golfer's elbow) on chin-ups and straight-bar curls",
"lateral elbow stinging on heavy tricep skull crushers",
"AC joint clicking and pain during dips",
"hamstring tightness when reaching bottom of Romanian deadlifts",
"wrist ache during front rack barbell squats"
]
PLATEAU_SEEDS = [
"stuck at 100kg bench press for 4 consecutive weeks on 5x5",
"squat e1RM has stalled for 3 sessions in a row",
"did 24 hard chest sets this week and recovery score is at 20%",
"overhead press hasn't progressed in a month, hitting volume ceiling",
"stuck on 8 reps with 30kg dumbbell press for past 3 workouts"
]
CONSTRAINT_SEEDS = [
"traveling in hotel gym with only dumbbells up to 20kg and a pullup bar",
"only have 35 minutes today, need a fast upper body hypertrophy session",
"home gym with only resistance bands and 16kg kettlebell, need leg workout",
"wrist sprain so cannot grip heavy barbells, need chest and back workout"
]
COACHING_SEEDS = [
"where should the barbell touch my chest on a flat bench press?",
"what does RPE 8 actually mean on heavy squats?",
"how wide should my grip be on lat pulldowns for max lat engagement?",
"should I lock out my knees completely at the top of a leg press?"
]

SYSTEM_INSTRUCTION = """
You are an expert dataset engineer building training data for an Agentic AI
Gym Coach.
Generate a batch of 100 realistic training conversations in valid JSON
format according to these EXACT rules:
SCHEMA PER CONVERSATION:
{
"messages": [
{"role": "system", "content": "You are an elite Gym Coach AI. You
manage user programming, injuries, and progressive overload. You have
access to tools: 'search_medical_db(query)',
'search_exercise_catalog(muscle, equipment, exclude_mechanics)', and
'generate_workout(workout_json)'. If an injury, pain, or plateau flag is
present, you MUST output a tool call in strict JSON format. If the user
asks for direct technique or RPE coaching, respond with direct coaching
text without tools."},
{"role": "user", "content": "<Lifter query using authentic gym slang>"},
{"role": "assistant", "content": "<Assistant output: EITHER a JSON tool
call string OR direct coaching text>"}
]
}

BATCH DISTRIBUTION (100 Items Total):- 40 Items: Joint pain/injuries. Assistant content MUST be: {\"thought\":
\"<reasoning>\", \"tool_call\": {\"name\": \"search_medical_db\",
\"arguments\": {\"query\": \"<search query>\"}}}- 30 Items: Plateaus/overtraining. Assistant content MUST be: {\"thought\":
\"<reasoning>\", \"tool_call\": {\"name\": \"search_medical_db\",
\"arguments\": {\"query\": \"<search query>\"}}}- 20 Items: Constraints/time crunch. Assistant content MUST be:
{\"thought\": \"<reasoning>\", \"tool_call\": {\"name\":
\"search_exercise_catalog\", \"arguments\": {<arguments>}}}- 10 Item: Direct form/RPE advice. Assistant content MUST be direct coaching
text (NO JSON, NO tool calls).
"""

all_samples = []
TARGET_COUNT = 800
print(f"[*] Starting automated dataset generation. Target: {TARGET_COUNT} samples.")
while len(all_samples) < TARGET_COUNT:
    # Build randomized context prompt
    batch_prompt = f"{SYSTEM_INSTRUCTION}\n\nSeed context for this batch:\n"
    batch_prompt += f"- Injuries: {random.sample(INJURY_SEEDS, 8)}\n"
    batch_prompt += f"- Plateaus: {random.sample(PLATEAU_SEEDS, 5)}\n"
    batch_prompt += f"- Constraints: {random.sample(CONSTRAINT_SEEDS, 4)}\n"
    batch_prompt += f"- Coaching: {random.sample(COACHING_SEEDS, 3)}\n"

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=batch_prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        batch = json.loads(response.text)
        if isinstance(batch, list):
            all_samples.extend(batch)
            print(f"[*] Collected: {len(all_samples)} / {TARGET_COUNT} samples.")
        time.sleep(12)
    except Exception as e:
        print(f"[!] Warning: Generation error ({e}). Retrying in 4 seconds...")
        time.sleep(10)
# Save output to JSONL
OUTPUT_PATH = "train_data.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in all_samples[:TARGET_COUNT]:
        f.write(json.dumps(item) + "\n")

print(f"[SUCCESS] Saved {TARGET_COUNT} training samples to {OUTPUT_PATH}")

[*] Starting automated dataset generation. Target: 800 samples.
[*] Collected: 109 / 800 samples.
[*] Collected: 219 / 800 samples.
[*] Collected: 338 / 800 samples.
[*] Collected: 440 / 800 samples.
[*] Collected: 548 / 800 samples.
[*] Collected: 668 / 800 samples.
[*] Collected: 778 / 800 samples.
[*] Collected: 907 / 800 samples.
[SUCCESS] Saved 800 training samples to train_data.jsonl


In [22]:
import json
valid_count = 0
tool_call_count = 0
direct_text_count = 0
with open("train_data.jsonl", "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        try:
            row = json.loads(line)
            assert "messages" in row, "Missing messages key"
            assert len(row["messages"]) == 3, "Must have exactly 3 message turns"
            assert row["messages"][0]["role"] == "system"
            assert row["messages"][1]["role"] == "user"
            assert row["messages"][2]["role"] == "assistant"
            # Check content type
            assistant_content = row["messages"][2]["content"]
            if "tool_call" in assistant_content:
                tool_call_count += 1
            else:
                direct_text_count += 1
            valid_count += 1
        except Exception as e:
            print(f"[!] Formatting Error on line {idx + 1}: {e}")

print(f"[*] Audit Results: {valid_count} valid rows.")
print(f"[*] Tool-Call Rows: {tool_call_count} | Direct Coaching Rows:{direct_text_count}")
if valid_count >= 800:
    print("[PASS] DATASET FULLY APPROVED FOR TRAINING.")

[*] Audit Results: 907 valid rows.
[*] Tool-Call Rows: 827 | Direct Coaching Rows:80
[PASS] DATASET FULLY APPROVED FOR TRAINING.


In [19]:
import json

# Save whatever is currently in memory to train_data.jsonl
OUTPUT_PATH = "train_data.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in all_samples:
        f.write(json.dumps(item) + "\n")

print(f"[SUCCESS] Saved {len(all_samples)} training samples to {OUTPUT_PATH}")

[SUCCESS] Saved 907 training samples to train_data.jsonl
